# 08 Copilot Research — M365 Copilot × 擬似GEPA でリサーチ→統合

調査依頼を投げると、**LocalLLM が目次（章立て）を設計** → 各章を **M365 Copilot（Researcher）経由で調査** →
**擬似GEPA でリサーチ指示を反復改良** → **1 本のレポートへ統合** する、ローカル Web アプリの使い方ノートです。

```
調査依頼 → 目次設計(LocalLLM) → ┌ 擬似GEPA: 章リサーチ(Copilot)→採点→内省→Pareto選択 ┐→ 確定リサーチ(全章) → 統合(LocalLLM) → レポート
                                └───────────── 予算(反復)ぶん繰り返す ─────────────┘
```

**M365 Copilot コネクタ（`connector`）は 4 種:**

| kind | 何をする | 実接続 |
|------|----------|--------|
| `demo` | モック（指示が濃いほど回答も濃くなる） | 不要（動作体験用） |
| `bridge` | 各章プロンプトを人が Copilot に貼り、回答を貼り戻す | ✅ 確実（自動化不可の社内環境向け） |
| `selenium` | **Edge（既定）/ Chrome** を WebDriver で実駆動（Researcher を自動操作） | ✅ 要ログイン・要調整 |
| `graph` | 任意 HTTP（Work IQ Chat API / 社内プロキシ）へ Bearer POST | ✅ 要エンドポイント・要トークン |

> このノートの**コードセルはすべて `demo` で動く**ので、LocalLLM も M365 も未接続のまま一通り試せます。

## 0. インストール（初回のみ）

```bash
cd jupyter-local-llm
pip install -e .            # アプリ本体（標準ライブラリのみで動作）
pip install selenium       # selenium コネクタを使う場合のみ
```
`demo` / `bridge` は追加インストール不要。`graph` は `httpx`（本体依存に含む）を使います。

## 1. アプリを起動（Web UI が主役）

`launch_copilot_research()` でバックグラウンド起動し、表示された URL をブラウザで開きます。

In [ ]:
import llmlab

url = llmlab.launch_copilot_research()   # 既定 http://127.0.0.1:8767
print("ブラウザで開く:", url)

開いたら:

1. **調査依頼** を入力
2. 実行モードで **デモ / 本番** を選ぶ（デモは接続不要）
3. **M365 Copilot コネクタ** を選択（bridge / selenium / graph）。`接続テスト` で疎通確認
4. **擬似GEPA 設定**（進化予算 / minibatch）
5. `① 目次を作成`（章立てを確認・編集）→ `② リサーチ実行`

進捗・GEPA の系譜・Pareto・スコア推移・章別の出典/採点・統合レポートがリアルタイムに表示されます。
レポートは **コピー / .md ダウンロード**、**実行履歴**から再表示できます。

## 2. 接続設定（本番モードで使う LocalLLM）

目次生成・採点・内省・統合は **LocalLLM（OpenAI 互換エンドポイント）** を使います。
Web UI 右上の `CONNECT` からでも、下記のコードでも設定できます（デモだけなら不要）。

In [ ]:
# 本番モードで使う場合のみ（デモ実行なら不要）
# llmlab.configure(
#     base_url="http://localhost:8000/v1",
#     api_key="...",
#     model="your-chat-model",
# )
llmlab.is_configured()

## 3. デモをコードから一気に実行（接続不要）

Web UI と同じパイプラインを、ローカルの HTTP API 経由で回す小さなヘルパーです。
`urllib`（標準ライブラリ）だけで動きます。

In [ ]:
import json, urllib.request

BASE = url  # 上で起動した URL（http://127.0.0.1:8767）

def _post(path, obj):
    req = urllib.request.Request(BASE + path, data=json.dumps(obj).encode(),
                                 headers={"Content-Type": "application/json"}, method="POST")
    return json.loads(urllib.request.urlopen(req, timeout=30).read())

def run_research(request, *, demo=True, connector="demo", connector_options=None,
                 gepa_budget=3, minibatch=2, outline=None, on_event=None):
    """1 回のリサーチを実行し、統合レポート（Markdown 文字列）を返す。"""
    payload = {"request": request, "demo": demo, "connector": connector,
               "connector_options": connector_options or {}, "gepa_budget": gepa_budget,
               "minibatch": minibatch, "outline": outline}
    run_id = _post("/api/run", payload)["run_id"]
    report = None
    stream = urllib.request.urlopen(BASE + "/api/events?id=" + run_id, timeout=600)
    for raw in stream:                       # SSE を逐次読む
        line = raw.decode().strip()
        if not line.startswith("data:"):
            continue
        e = json.loads(line[5:].strip())
        if on_event:
            on_event(e)
        if e["type"] == "final":
            report = e["report"]
        if e["type"] == "done":
            break
    return report

In [ ]:
def show(e):
    t = e["type"]
    if t == "outline":
        print("目次:", e["title"], "::", [c["title"] for c in e["chapters"]])
    elif t == "gepa_candidate":
        print(f"  候補 {e['id']} agg={e['agg']:.3f} {'[Pareto]' if e['pareto'] else ''}  {e['rationale']}")
    elif t == "gepa_best":
        print("  ベスト:", e["id"], "agg=", round(e["agg"], 3))
    elif t == "chapter" and e.get("phase") == "final":
        print(f"  章『{e['title']}』 score={e.get('score')}  出典 {len(e.get('citations', []))} 件")

report = run_research("社内文書検索に RAG を導入すべきか（費用対効果・リスク・他社事例）",
                      demo=True, gepa_budget=3, minibatch=2, on_event=show)
print("\n" + "=" * 60)
print(report)

擬似GEPA が反復ごとに指示へ「出典の明示」「数値で定量化・比較」「最新動向」を足していき、
`agg`（総合スコア）が上がっていくのが見えます。最後に**勝ち残った指示で全章を確定リサーチ**し、統合します。

## 4. 目次だけ先に作る（人手編集の前段）

`propose_outline()` で章立てだけ取得できます（`demo=True` は LLM 不要）。
編集した目次を `run_research(..., outline=...)` に渡せば、その章立てで実行します。

In [ ]:
from llmlab import copilotresearch as cr

outline = cr.propose_outline("生成AIの社内活用ロードマップ", demo=True)
print(outline["title"])
for i, ch in enumerate(outline["chapters"], 1):
    print(f"  {i}. {ch['title']} — {ch['goal']}")

# 例: 章を1つ足して、その目次で実行（demo）
outline["chapters"].append({"title": "投資対効果とKPI", "goal": "ROIと評価指標を定義する"})
# report2 = run_research("生成AIの社内活用ロードマップ", demo=True, outline=outline, on_event=show)

## 5. M365 Copilot コネクタを直接使う

各コネクタは `m365copilot.make_connector(kind, options)` で作れます。
`.test()` で設定チェック、`.research(prompt, meta=...)` で 1 章分の調査を実行します。

In [ ]:
from llmlab import m365copilot as mc

print("利用可能:", [c["kind"] for c in mc.connector_kinds()])

# demo コネクタで 1 章だけ調べてみる
demo = mc.make_connector("demo")
r = demo.research("背景と全体像を、出典つき・数値つき・比較つきで調べて",
                  meta={"chapter": "背景と全体像", "topic": "RAG導入", "idx": 0})
print("ok:", r.ok, "/ 出典:", r.citations)
print(r.text)

### 5-1. Selenium（Edge 既定）で実接続 — ドライバの場所とウェイト

M365 Copilot は **Edge + 業務アカウントの SSO** で使うことが多いので既定は Edge です。
**WebDriver（msedgedriver）の場所は `driver_path` で明示指定**できます（環境変数 `EDGEDRIVER_PATH` でも可）。

主な options:

| キー | 意味 |
|------|------|
| `browser` | `"edge"`（既定）/ `"chrome"` |
| `driver_path` | msedgedriver / chromedriver の場所（空なら自動解決） |
| `browser_binary` | ブラウザ実行ファイルの場所（空なら既定） |
| `user_data_dir` | ログイン状態を保存するプロファイル（初回ログインを再利用） |
| `headless` | 無人実行（**初回ログインは False**） |
| `agent_selector` | Researcher 選択チップの CSS（指定で自動選択） |
| `busy_selector` | **「生成中」インジケータの CSS（表示中は待機を継続）** |
| `ready_timeout_ms` / `initial_wait_ms` / `poll_ms` / `settle_ms` / `timeout_ms` | 待機（ウェイト）調整。Researcher は長いので `timeout_ms` を大きめに |

> 初回は `headless=False` で一度ログインし、以降は同じ `user_data_dir` で無人運用できます。

In [ ]:
edge_opts = {
    "browser": "edge",
    "driver_path": r"C:\tools\msedgedriver.exe",   # ← ドライバの場所を明示指定（環境に合わせて）
    # "browser_binary": r"C:\Program Files (x86)\Microsoft\Edge\Application\msedge.exe",
    "headless": False,                                # 初回ログインは False
    "agent_selector": "button[aria-label*='Researcher']",   # Researcher を自動選択（任意）
    "busy_selector": "button[aria-label*='停止'], [class*='typing']",  # 生成中は待機継続
    "timeout_ms": 600000,                             # Researcher は長い → 10 分待つ例
}

print(mc.make_connector("selenium", edge_opts).test())   # selenium 未導入なら導入を促すメッセージ

# 本番実行の例（LocalLLM 接続 + Edge ログイン済みが前提）:
# report = run_research("最新のAI規制動向", demo=False, connector="selenium",
#                       connector_options=edge_opts, gepa_budget=2, minibatch=2, on_event=show)

### 5-2. 人手ブリッジ（bridge）— 自動化なしで確実

自動化や API が塞がれた社内環境向け。各章のプロンプトが Web UI に表示されるので、
**人が M365 Copilot（Researcher）に貼り付け → 回答を貼り戻す**だけ。API 提供状況に依存せず確実に動きます。
（この方式は対話が必要なため Web UI から使ってください。）

### 5-3. graph（HTTP / Work IQ Chat API・社内プロキシ）

任意の HTTP エンドポイントへ Bearer トークンで POST します。要求/応答の JSON 形状は設定で吸収します。
Microsoft の **Work IQ Chat API**（Copilot の応答を出典つきで返す）や、社内 Copilot プロキシに向けられます。

In [ ]:
graph_opts = {
    "endpoint": "https://your-copilot-endpoint/chat",  # ← あなたのエンドポイント
    "token_env": "M365_COPILOT_TOKEN",                 # 環境変数からトークンを読む
    "prompt_field": "message",                          # リクエストのプロンプト格納キー
    "answer_path": "choices.0.message.content",         # 応答から回答を取り出すJSONパス
}
print(mc.make_connector("graph", graph_opts).test())   # endpoint/token 未設定なら不足を教えてくれる

## 6. 擬似GEPA のパラメータと読み方

- **進化予算 `gepa_budget`**: 指示を改良する反復回数（0 = 進化なし＝シード指示のまま）。
- **`minibatch`**: 探索中に評価する代表章の数（少ないほど Copilot 呼び出し＝コストが少ない）。
- **採点**: 各章を `網羅性 / 具体性 / 出典 / 依頼適合` の 4 観点で 0〜1 に採点（本番は LLM 審査）。
- **Pareto 選択**: 章ごとのスコア・ベクトル上で非劣な候補（フロンティア）から次の親を選ぶので、
  「ある章だけ強い候補」も生き残り、多様性を保ちます（UI に系譜・Pareto バッジ・スコア曲線を表示）。
- **最終**: 予算到達後、総合が最良の指示で**全章を確定リサーチ**し統合します。

> `bridge`（人手貼付）は 1 章ごとに貼り戻しが要るため、`gepa_budget` は小さめ（0〜1）が実用的。
> `demo` / `selenium` / `graph` は自動なので予算を大きくできます。

## 7. 出力の保存先・実行履歴

実行結果（レポート全文つき）は `~/.llmlab/copilot/runs.json` に保存され、Web UI の**実行履歴**から再表示できます。
コードからは `runs_all()` で参照できます。

In [ ]:
runs = cr.runs_all()
print("保存されている実行:", len(runs))
for r in runs[:3]:
    g = r.get("gepa", {})
    print(f"  {r['ts']} [{r['status']}] {r['connector']}{'(demo)' if r['demo'] else ''} "
          f"agg={g.get('best_agg', 0):.2f} :: {r['title']}")

## まとめ

- **手軽に試す**: `llmlab.launch_copilot_research()` → ブラウザで **デモ実行**。
- **本番**: `CONNECT` で LocalLLM を設定 → コネクタを選ぶ
  （社内で確実なら **bridge**、自動化なら **selenium(Edge)** か **graph**）。
- **Researcher は時間がかかる**ので selenium は `busy_selector` と大きめの `timeout_ms` を設定。
- **同時実行**は各 run 独立（selenium を同時に使う時だけ run ごとに別 `user_data_dir` を指定）。

詳細は `jupyter-local-llm/README.md`（⑫ Copilot Research）を参照してください。